# SInERT — Entraînement des 10 modèles restants (Colab)

**Modèles couverts :** SARIMAX · XGBoost · LightGBM · SVR · LSTM · Bi-LSTM · GRU · CNN-1D · TCN/ResNet-1D · Transformer · CNN-LSTM

**Protocole identique au benchmark local :** entrée = semaine écoulée (672×11), cible = jour suivant (96×4 : ΔEa, ΔEr, Pmax, Smax) — split train/val/test chronologique déjà encodé dans le fichier de données.

**À faire :**
1. Exécuter les cellules dans l'ordre. Les résultats **s'accumulent** entre sessions (si la session tombe, relancez simplement la cellule concernée).
2. Conseillé : *Exécution > Modifier le type d'exécution > GPU* avant l'étape 2 (DL). Les étapes 1 tournent bien sur CPU.
3. À la fin : télécharger `colab_results.csv` + `colab_predictions.npz` et les remettre dans `output/benchmark/` du projet, puis lancer `python -I phase2/merge_colab.py` en local.

In [ ]:
from google.colab import files
print('Choisissez le fichier windows_scaled.npz (~16 Mo)')
up = files.upload()
NPZ = list(up.keys())[0]
assert NPZ.endswith('.npz'), 'il faut windows_scaled.npz'
print('OK ->', NPZ)

In [ ]:
# récupérer le code du benchmark depuis le dépôt GitHub (public)
!wget -q -O colab_benchmark.py https://raw.githubusercontent.com/tfiras11/sinert-energy-benchmark/main/phase2/colab_benchmark.py
import py_compile; py_compile.compile('colab_benchmark.py', doraise=True)
print('colab_benchmark.py prêt')

In [ ]:
import torch
print('Device :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
if not torch.cuda.is_available():
    print("(Pour l'étape 2, activez le GPU : Exécution > Modifier le type d'exécution > GPU)")

## Étape 1 — Machine Learning + SARIMAX (~20–35 min, CPU)

XGBoost → LightGBM → SVR → SARIMAX (walk-forward : prévision jour *d*, puis injection des réalisations dans le filtre de Kalman).

In [ ]:
!python colab_benchmark.py --data {NPZ} --models ml,sarimax

## Étape 2 — Deep Learning (~20–30 min, GPU)

LSTM, Bi-LSTM, GRU, CNN-1D, TCN/ResNet-1D (blocs résiduels + BatchNorm), Transformer, CNN-LSTM — tête multi-sorties 96×4, early stopping sur la validation (patience 7).

In [ ]:
!python colab_benchmark.py --data {NPZ} --models dl --epochs 60

## Résultats

In [ ]:
import pandas as pd
df = pd.read_csv('colab_results.csv')
cols = ['model','family','n_params','train_time_s','RMSE_dEa_kWh','MAPE_dEa_kWh_daily','R2_dEa_kWh','PDE_Pmax','PDE_Smax']
df[cols].sort_values('RMSE_dEa_kWh').round(3)

In [ ]:
from google.colab import files
files.download('colab_results.csv')
files.download('colab_predictions.npz')

## Ensuite (en local)

1. Déposer les 2 fichiers dans  du projet
2. `python -I phase2/merge_colab.py` → `benchmark_results.csv` avec les 13 modèles
3. Commit + push GitHub, puis Phases 3-5 (voir HANDOFF.md)